# Setup
## Dependencies

In [1]:
from pathlib import Path
import sqlite3
from tqdm.auto import tqdm

## SQLite Setup

In [2]:
exposure_db_path = Path("/Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/databases/Durham_Observation_Database.db").resolve()
exposure_conn = sqlite3.connect(exposure_db_path)

master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

In [3]:
master_conn.execute("""
   CREATE TABLE IF NOT EXISTS observations (
       object TEXT,
       obs_date TEXT,
       telescope TEXT,
       img_start TEXT NOT NULL,
       img_end TEXT NOT NULL,
       num_exps INTEGER,
       filter TEXT,
       exp_time REAL,
       mosaic_name_f TEXT,
       zpt_mean_f REAL,
       zpt_median_f REAL,
       zpt_mean_weighted_f REAL,
       zpt_stdev_f REAL,
       ztp_err_f REAL,
       mag_f REAL,
       mag_err_f REAL,
       mosaic_name TEXT,
       zpt_mean REAL,
       zpt_median REAL,
       zpt_mean_weighted REAL,
       zpt_stdev REAL,
       ztp_err REAL,
       mag REAL,
       mag_err REAL,
       error_f TEXT,
       error TEXT,
       url_source TEXT NOT NULL,
       group_id INTEGER PRIMARY KEY,
       UNIQUE (url_source, img_start, img_end)     
   )
""")

## Defining Functions

In [4]:
def get_target_details(conn, target_name):
    sql = """
        SELECT
            UPPER(TRIM(object)) AS target,
            TRIM(obs_date) AS date,
            TRIM(telescope) AS telescope,
            TRIM(img_num) AS img_num,
            TRIM(filter) AS filter,
            (exp_time) AS exp_time,
            TRIM(start_time) AS start_time,
            (fwhm) AS fwhm,
            TRIM(url_source) AS url
        FROM observations
        WHERE UPPER(TRIM(object)) = UPPER(?)
        ORDER BY obs_date DESC, telescope, CAST(img_num AS INTEGER) 
    """
    return conn.execute(sql, (target_name,)).fetchall()

In [5]:
def get_target_list(exposure_conn):
    sql = """
        SELECT
            CASE
                WHEN UPPER(TRIM(object)) LIKE 'SN%'
                    THEN 'SN' || LOWER(SUBSTR(TRIM(object), 3))
                ELSE 'AT' || LOWER(SUBSTR(TRIM(object), 3))
            END AS target
        FROM observations
        WHERE UPPER(TRIM(object)) LIKE 'SN%'
            OR UPPER(TRIM(object)) LIKE 'AT%'
        GROUP BY UPPER(TRIM(object))
        HAVING MAX(obs_date) >= "2026-07-01"
        ORDER BY target
    """

    temp_rows = exposure_conn.execute(sql).fetchall()
    #print(temp_rows)

    list_of_supernovae = []
    for row in temp_rows:
        object_name = row[0]
        #print(object_name)
        list_of_supernovae.append(object_name)

    return(list_of_supernovae)

In [6]:
def get_image_ranges(exposures):
    numbers = sorted({row[3] for row in exposures}, key=int)

    if not numbers:
        return []

    ranges = []
    start = numbers[0]
    previous = numbers[0]

    for number in numbers[1:]:
        if int(number) != int(previous) + 1:
            num_exps = int(previous) - int(start) + 1
            ranges.append((start, previous, num_exps))

            start = number

        previous = number

    num_exps = int(previous) - int(start) + 1
    ranges.append((start, previous, num_exps))

    return ranges

In [7]:
def get_exposure_groups(list_of_supernovae, exposure_conn):
    exposure_groups = {}

    for targets in list_of_supernovae:
        exposure_rows = get_target_details(exposure_conn, targets)

        for row in exposure_rows:
            (object_name, obs_date, telescope,
             img_num, band, exp_time, 
             start_time, fwhm, url_source,
            ) = row
        
            #to turn them into groups, which values they share
            
            group_params = (
                object_name, obs_date, telescope,
                band, exp_time, url_source
            )
        
            if group_params not in exposure_groups:
                   exposure_groups[group_params] = []
        
            exposure_groups[group_params].append(row)

    return exposure_groups

    

In [10]:
def insert_exposure_groups(master_conn, exposure_conn):
    sql = """
        INSERT INTO observations (
            object, obs_date, telescope,
            img_start, img_end, num_exps,
            filter, exp_time, url_source
           )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT (url_source, img_start, img_end) DO NOTHING
    """

    inserted =0
    
    with master_conn:
        for group_params, exposures in tqdm(
            exposure_groups.items(),
            desc="Processing exposure groups",
            unit="group",
        ):
            (object_name, obs_date, telescope,
             band, exp_time, url_source
            ) = group_params

            image_ranges = get_image_ranges(exposures)

            for img_start, img_end, num_exps in image_ranges:
                cursor = master_conn.execute(sql, (
                    object_name, obs_date, telescope,
                    img_start, img_end, num_exps,
                    band, exp_time, url_source,
                ))

                inserted += cursor.rowcount

    return inserted

# The Actual Execution Cell

In [11]:
list_of_supernovae = get_target_list(exposure_conn)
exposure_groups = get_exposure_groups(list_of_supernovae, exposure_conn)
insert_exposure_groups(master_conn, exposure_groups)

Processing exposure groups:   0%|          | 0/192 [00:00<?, ?group/s]

194